# CPSA 03 — FP32 training on 2× T4

YOLOv8-pose with Hard-Swish, no attention. Ultralytics trains square
`imgsz=640` with `rect=False`; the deployment graph is exported and audited
at the camera's native 480×640 input.


In [ ]:
import subprocess
import sys

# Pin the exact CPSA_2026 commit for a reproducible Kaggle run.
COMMIT_SHA = "REPLACE_WITH_COMMIT_SHA"

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        f"git+https://github.com/fcabecciaw/CPSA_2026@{COMMIT_SHA}#subdirectory=ml",
    ],
    check=True,
)

In [ ]:
from pathlib import Path

from cpsa_ml.train import TrainConfig, train

DATA = Path("/kaggle/input/cpsa-halfbust/data.yaml")
SCHEMA = "halfbust62"
VARIANT = "n"
EPOCHS = 100
BATCH = 32
DEVICE = "0,1"  # 2× T4 DDP
IMGSZ = 640
PROJECT = Path("/kaggle/working/runs")
NAME = "halfbust62-yolov8n-hswish-fp32"
RESUME = ""  # e.g. "/kaggle/input/cpsa-checkpoints/last.pt"

best = train(
    TrainConfig(
        data=DATA,
        schema=SCHEMA,
        variant=VARIANT,
        epochs=EPOCHS,
        batch=BATCH,
        device=DEVICE,
        imgsz=IMGSZ,
        project=PROJECT,
        name=NAME,
        resume=Path(RESUME) if RESUME else None,
        seed=0,
        workers=4,
        amp=True,
        val=True,
        plots=True,
    )
)
print("best checkpoint:", best)

In [ ]:
from cpsa_ml.onnx_check import check_onnx, export_onnx, format_report

ONNX = Path("/kaggle/working/export/halfbust62.onnx")
onnx_path = export_onnx(best, ONNX, imgsz=(480, 640))
report = check_onnx(onnx_path, imgsz=(480, 640), opset=17)
print(format_report(report))
assert report.passed, "backbone/neck is not DPU-compatible"

In [ ]:
# Re-publish /kaggle/working/runs as a dataset version. In the next session set
# RESUME to the attached .../weights/last.pt.
for path in sorted((PROJECT / NAME / "weights").glob("*.pt")):
    print(path)